# Third PyCUDA example: vector addition with GPUArray

This is the third notebook in the five-example PyCUDA sequence. It uses the **same input values, vector length and single precision** as the first two notebooks to compute $c_i=a_i+b_i$.

The new interface is `pycuda.gpuarray.GPUArray`: arrays stored on the GPU support arithmetic expressions such as `d_c = d_a + d_b`. PyCUDA handles the device kernel and its launch, so this notebook needs neither user-written CUDA source nor an explicit block/grid configuration.

Run the cells in order in a GPU runtime. Validation, selected results and optional timings appear directly in the notebook; no result files are exported.


## 1. Prepare the GPU runtime

In Google Colab, select **Runtime → Change runtime type → GPU** and connect. The following commands inspect the assigned GPU and CUDA compiler on the remote machine. `SourceModule` needs `nvcc`; a GPU runtime alone does not establish that the compiler is available.


In [1]:
!nvidia-smi
!nvcc --version


Wed Oct 14 17:30:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Install PyCUDA if necessary. The installation may build the package and take several minutes. Continue only after installation succeeds.


In [2]:
!pip install pycuda


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 36.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 11.2 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp313-cp313-linux_x86_64.whl size=5281393 sha256=e23d252c0138fd2a6fc8346b13c4b539841d116a3b1f6dbce32b30343cd54bb8
  Stored in directory: /root/.cache/pip/wheels/ce/26/46/c519675fcb0e5e17bab8e85b6676528c40d12d794182340e85
Successfully built pycuda


## 2. Initialize PyCUDA

`pycuda.autoinit` creates the CUDA context. Import the `gpuarray` module to create and operate on `GPUArray` objects. We retain the environment printout used in the first two examples. A working CUDA compiler is still needed for kernels generated by PyCUDA, even though we do not write their source ourselves.


In [3]:
import sys
import numpy as np
import pycuda
import pycuda.driver as cuda
import pycuda.autoinit
import pycuda.gpuarray as gpuarray

device = cuda.Context.get_device()
print('GPU:', device.name())
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('PyCUDA:', pycuda.VERSION_TEXT)


GPU: Tesla T4
Python: 3.13.15
NumPy: 2.1.3
PyCUDA: 2026.1


## 3. Choose the vector length

Use $N=100003$, as in the first two examples. Here PyCUDA determines the launch configuration and handles the full array length internally. There is no user-defined `BLOCKSIZE`, `grid`, or rounding function.


In [4]:
N = 100003
assert 1 <= N <= np.iinfo(np.int32).max
print(f'Elements: {N}')
print('Launch configuration: managed by GPUArray.')


Elements: 100003
Launch configuration: managed by GPUArray.


## 4. Prepare the host arrays

Use the same deterministic inputs as before: $a_i=0.25(i\bmod101)$ and $b_i=0.50(i\bmod37)$. Both arrays have type `np.float32`. We allocate `h_c` on the host for the result to be retrieved later.


In [5]:
indices = np.arange(N, dtype=np.int32)
h_a = np.float32(0.25) * (indices % 101).astype(np.float32)
h_b = np.float32(0.50) * (indices % 37).astype(np.float32)
h_c = np.empty_like(h_a)

assert h_a.dtype == h_b.dtype == h_c.dtype == np.float32
assert all(x.flags.c_contiguous for x in (h_a, h_b, h_c))
print(f'Bytes per vector: {h_a.nbytes}')


Bytes per vector: 400012


## 5. Create GPU arrays and transfer the inputs

`gpuarray.to_gpu` allocates device storage and copies the host data into it. The returned objects contain array information such as shape and dtype as well as their device storage. Unlike the temporary argument handlers of the second example, `d_a` and `d_b` remain available for subsequent GPU operations.

The printout below shows metadata only; it does not retrieve the array values.


In [6]:
d_a = gpuarray.to_gpu(h_a)
d_b = gpuarray.to_gpu(h_b)

print('Device array class:', type(d_a).__name__)
print('Input shape:', d_a.shape)
print('Input dtype:', d_a.dtype)
print('Device bytes per input:', d_a.nbytes)


Device array class: GPUArray
Input shape: (100003,)
Input dtype: float32
Device bytes per input: 400012


## 6. Add the arrays on the GPU

Because `d_a` and `d_b` are `GPUArray` objects, their `+` operator performs elementwise addition on the device. The expression creates a new GPU array `d_c`; it does not overwrite the inputs or copy the result back to the host.

The numerical operation is unchanged, but we are **not compiling the `vector_add` source from the preceding notebooks**. PyCUDA supplies the elementwise implementation and selects its launch configuration. The first use can involve kernel compilation. GPU work may still be pending after this Python statement returns; the following blocking retrieval makes the result available to the CPU.


In [7]:
d_c = d_a + d_b


## 7. Retrieve and validate every output

The blocking `get` method copies the device result into the existing host array `h_c`. Without the `ary` argument, `get()` would create and return a new NumPy array instead. Use exactly the same reference calculation, finite-value check and tolerances as before.


In [8]:
d_c.get(ary=h_c)
h_reference = h_a + h_b

if not np.all(np.isfinite(h_c)):
    raise AssertionError('The GPU result contains nonfinite values.')
np.testing.assert_allclose(h_c, h_reference, rtol=1e-6, atol=1e-6)
max_abs_error = float(np.max(np.abs(h_c - h_reference)))
print(f'Validation passed for all {N} elements.')
print(f'Maximum absolute error: {max_abs_error:.3e}')


Validation passed for all 100003 elements.
Maximum absolute error: 0.000e+00


## 8. Display the computed sum

The `c[i] (GPU)` column contains the values retrieved from `d_c`. Display the first twelve and the last three entries, using the same layout as the previous notebooks.


In [9]:
shown_indices = sorted(set(range(min(12, N))) | set(range(max(0, N - 3), N)))
print(f"{'i':>8} {'a[i]':>12} {'b[i]':>12} {'c[i] (GPU)':>15} {'NumPy':>12}")
for i in shown_indices:
    print(f'{i:8d} {h_a[i]:12.6f} {h_b[i]:12.6f} '
          f'{h_c[i]:15.6f} {h_reference[i]:12.6f}')


       i         a[i]         b[i]      c[i] (GPU)        NumPy
       0     0.000000     0.000000        0.000000     0.000000
       1     0.250000     0.500000        0.750000     0.750000
       2     0.500000     1.000000        1.500000     1.500000
       3     0.750000     1.500000        2.250000     2.250000
       4     1.000000     2.000000        3.000000     3.000000
       5     1.250000     2.500000        3.750000     3.750000
       6     1.500000     3.000000        4.500000     4.500000
       7     1.750000     3.500000        5.250000     5.250000
       8     2.000000     4.000000        6.000000     6.000000
       9     2.250000     4.500000        6.750000     6.750000
      10     2.500000     5.000000        7.500000     7.500000
      11     2.750000     5.500000        8.250000     8.250000
  100000     2.500000    13.000000       15.500000    15.500000
  100001     2.750000    13.500000       16.250000    16.250000
  100002     3.000000    14.000000      

## 9. Optional: measure repeated GPUArray additions

Keep both inputs on the GPU and perform three warm-up additions before timing. A host timer then measures 100 evaluations of `d_c = d_a + d_b`, followed by a wait for GPU completion.

This interval includes Python overhead, allocation of successive output arrays, any release of replaced outputs during the loop, kernel submission, execution and the final wait. Input allocation and transfers, compilation during warm-up, output retrieval and validation are excluded. The final output remains allocated after timing.

Report the mean interval per expression evaluation. This is **not isolated kernel time** and is **not the complete host-to-device/add/device-to-host operation**. The first notebook times repeated kernel launches; the second times complete calls including transfers. Their printed numbers therefore cannot be used directly to rank the three interfaces.


In [10]:
from time import perf_counter

REPEATS = 100
assert REPEATS > 0

for _ in range(3):
    d_c = d_a + d_b
cuda.Context.synchronize()

start = perf_counter()
for _ in range(REPEATS):
    d_c = d_a + d_b
cuda.Context.synchronize()
elapsed_ms = (perf_counter() - start) * 1000.0
mean_ms = elapsed_ms / REPEATS

# Retrieve and validate the final result outside the timed interval.
d_c.get(ary=h_c)
if not np.all(np.isfinite(h_c)):
    raise AssertionError('The GPU result contains nonfinite values.')
np.testing.assert_allclose(h_c, h_reference, rtol=1e-6, atol=1e-6)

print(f'Repeated GPUArray additions: {REPEATS}')
print(f'Total host elapsed time: {elapsed_ms:.6f} ms')
print(f'Mean time per GPUArray addition: {mean_ms:.6f} ms')
print('Includes output allocation, Python overhead, GPU work and final wait.')
print('Excludes host-device transfers; this is not isolated kernel time.')
print('Final result validated.')


Repeated GPUArray additions: 100
Total host elapsed time: 3.971260 ms
Mean time per GPUArray addition: 0.039713 ms
Includes output allocation, Python overhead, GPU work and final wait.
Excludes host-device transfers; this is not isolated kernel time.
Final result validated.


## 10. Release the GPU arrays

The GPU arrays manage their device storage. Once GPU work is complete, remove the notebook's references when finished. In this standalone example there are no other references to these arrays, so their storage can be released automatically. Do not manually free `gpudata` while continuing to use its owning array. To repeat the computation after this cell, start again from the input-transfer cell.


In [11]:
cuda.Context.synchronize()
del d_a, d_b, d_c
print('GPU array references released.')


GPU array references released.


## What changed from the first two examples?

The first example uses an explicit CUDA kernel and explicit device buffers. The second retains that kernel and uses `cuda.In`/`cuda.Out` to handle its transfers. This third example expresses the sum directly through GPU array arithmetic. Device storage persists with the arrays, and retrieval remains an explicit operation.

The familiar syntax does not mean the addition runs in NumPy on the CPU: the operand types determine where it is executed. It also does not mean that `GPUArray` implements every NumPy operation.

**Execution status:** supplied with cleared outputs. Notebook structure and Python syntax were checked during preparation; GPU execution was not available in that environment. Run the cells on your GPU to obtain validation results and timings.

Reference: [PyCUDA GPUArray documentation](https://documen.tician.de/pycuda/array.html).
